# 🔌 Connect to Your RADKit Service

There are many different ways to connect to your RADKit service. Depending on your environment, you will choose one on top of another.

## What you will do
- Connect through Cisco Cloud with SSO (most common)
- Connect with certificate-based login (automation friendly)
- Connect directly to your server without cloud access

---

## 1) Load Local Configuration

Before connecting, load environment variables from your `.env` file.

Expected values:
- `RADKIT_USER`: your CCO user ID
- `RADKIT_SERVICE`: your RADKit service ID

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

---

## 2) Connect via Cisco Cloud (SSO Login)

**Best for**: Interactive use, development environments, and any scenario where a browser window can be opened.

**How it works**:

1. `sso_login` opens your web browser with the Cisco SSO login form.
2. The script pauses and waits until you complete the login.
3. Once authenticated, execution resumes automatically.

Click on the code block below. A web browser will be opened, prompting you for authentication.

Use your RADKit Remote User to authenticate. Once done, return to this notebook. You will see that you have an active connection to your RADKit service.

In [2]:
from radkit_client import Client
from radkit_client.sync import ClientStatus, ServiceStatus # Nice enums to get the status of the client and service

user_id = os.getenv("RADKIT_USER")
service_id = os.getenv("RADKIT_SERVICE")

with Client.create() as client:
    client.sso_login(user_id)
    print("✅ Authentication successful!") if client.status == ClientStatus.CONNECTED else print("🔥 Connection failed ...")
    
    service = client.service_cloud(service_id).wait()
    print(f"✅ Connection successful to service {service.service_id}!") if service.status == ServiceStatus.READY else print(f"🔥 Connection to service {service.service_id} failed ...")

<frozen radkit_common.utils.ssl>:518: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.



A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.
✅ Authentication successful!
✅ Connection successful to service 21km-e0xp-fcib!


---

## 3) Certificate Login (`certificate_login`)

**Best for:** Non-interactive workflows such as CI/CD, scheduled jobs, or headless scripts.

**Why choose this:** No browser prompt is required during login. Authentication uses locally stored client certificates.

**Important prerequisite:** You must enroll this machine/client first so certificate files exist.

---

Open a terminal and activate your virtual environment based on your computer's operating system type.

Once done, execute the following script and follow the instructions given:

```bash
python src/enroll-client.py
```

During enrollment, save your certificate private-key password. You will be prompted for it in the next cell.

After enrollment completes, run the next code cell.

In [ ]:
from radkit_client import Client
from radkit_client.sync import ClientStatus, ServiceStatus
import getpass

user_id = os.getenv("RADKIT_USER")
service_id = os.getenv("RADKIT_SERVICE")
private_key_password = getpass.getpass("🔑 Enter the password for your private key: ")

with Client.create() as client:
    client.certificate_login(identity=user_id, private_key_password=private_key_password)
    print("✅ Authentication successful!") if client.status == ClientStatus.CONNECTED else print("🔥 Connection failed ...")
    
    service = client.service_cloud(service_id).wait()
    print(f"✅ Connection successful to service {service.service_id}!") if service.status == ServiceStatus.READY else print(f"🔥 Connection to service {service.service_id} failed ...")

✅ Authentication successful!
✅ Connection successful to service 21km-e0xp-fcib!


---

## 4) Connect Directly (No Cloud)

**Best for:** Air-gapped environments, restricted outbound internet, or private-network-only deployments.

**Why choose this:** The client connects directly to the RADKit server endpoint over LAN/VPN, bypassing Cisco Cloud.

**You will need:**
- Your **CCO user ID**
- Your **E2EE validation token** (used as the password for direct auth)
- The server **hostname or IP address**
- The server **RPC port** (default: `8181`)

---

Before moving forward, you need to connect first to your VPN using the Cisco Anyconnect VPN client and the following information:

- `Domain`: Domain that you received via e-mail
- `Username`: The VPN username you received via e-mail
- `Password`: The VPN password you received via e-mail

Afterwards, execute the following code block. Use the E2EE validation token provided to you via e-mail.

In [2]:
from radkit_client import Client
from radkit_common.rpc.client_transports.verify import RPCVerificationError # Specific exception for failed RPC verification
import getpass

server_address ="10.10.20.59"
RPC_PORT = 8181

user_id = os.getenv("RADKIT_USER")
e2ee_validation_token = getpass.getpass("🔑 Enter your E2EE validation token: ")

with Client.create() as client:
    service = client.service_direct(
        username=user_id,
        host=server_address,
        port=int(RPC_PORT),
        password=e2ee_validation_token
    )
    try:
        service.wait()
        print(f"✅ Connection successful to service!")
    except RPCVerificationError as e:
        print(f"🔥 Connection to service failed: {e}")
    except Exception as e:
        print(f"🔥 An unexpected error occurred while connecting to service: {e}")

✅ Connection successful to service!
